In [1]:
from marker.converters.table import TableConverter
from marker.config.parser import ConfigParser
from marker.models import create_model_dict
from marker.output import text_from_rendered
import os

config = {
    "disable_image_extraction": True,
    "force_ocr": True,
    "strip_existing_ocr": True,
    "page_range": "10-24",
}

config_parser = ConfigParser(config)

converter = TableConverter(
    config=config_parser.generate_config_dict(),
    artifact_dict=create_model_dict(),
)

current_path = os.getcwd()
script_dir = os.path.dirname(current_path)
input_dir = os.path.join(script_dir, "datas\\SNI_7657-2023.pdf")
print(current_path)
print(input_dir)

rendered = converter(input_dir)
text, _, images = text_from_rendered(rendered)

`torch_dtype` is deprecated! Use `dtype` instead!


d:\Develop\repo\db-wilayah-indonesia\sources\notebooks
d:\Develop\repo\db-wilayah-indonesia\sources\datas\SNI_7657-2023.pdf


Recognizing Text: 100%|██████████| 3174/3174 [02:10<00:00, 24.36it/s]


In [3]:
print(text)

| No. | Provinsi       | Kabupaten/Kota                | Nama Kota             | Singkatan<br>Nama<br>Kota | Parent<br>subdivision<br>(ISO 3166-2:2020) |
|-----|----------------|-------------------------------|-----------------------|---------------------------|--------------------------------------------|
| 1.  | Aceh           | Kabupaten Aceh<br>Selatan     | Tapak Tuan            | TTN                       | ID-AC                                      |
| 2.  | Aceh           | Kabupaten Aceh<br>Tenggara    | Kutacane              | KTN                       | ID-AC                                      |
| 3.  | Aceh           | Kabupaten Aceh Timur          | ldi Rayeuk            | IRY                       | ID-AC                                      |
| 4.  | Aceh           | Kabupaten Aceh<br>Tengah      | Takengon              | TKN                       | ID-AC                                      |
| 5.  | Aceh           | Kabupaten Aceh Barat          | Meulaboh           

In [16]:
import polars as pl
from io import StringIO
import re

from utils.converter import convert_cyrillic_to_latin

table_md = '\n'.join([line for line in text.split('\n') if not line.startswith('|-----') and line.strip()])
df = pl.read_csv(StringIO(table_md), separator='|', has_header=True)
df = df.select([col for col in df.columns if col.strip()])
columns = df.columns
no_col = next(col for col in columns if 'No' in col)
prov_col = next(col for col in columns if 'Provinsi' in col)
kab_col = next(col for col in columns if 'Kabupaten' in col)
nama_col = next(col for col in columns if 'Nama Kota' in col)
singk_col = next(col for col in columns if 'Singkatan' in col)
parent_col = next(col for col in columns if 'Parent' in col)

cleaned_records = []
cyr_to_lat = {# Digits that look similar
        'О': '0', 'З': '3', 'Б': '8',
        # Uppercase letters
        'А': 'A', 'В': 'B', 'Е': 'E', 'К': 'K', 'М': 'M', 
        'Н': 'H', 'О': 'O', 'Р': 'P', 'С': 'C', 'Т': 'T',
        'Х': 'X', 'У': 'Y', 'І': 'I', 'Ї': 'I', 'Ё': 'E',
        # Lowercase letters  
        'а': 'a', 'в': 'b', 'е': 'e', 'к': 'k', 'м': 'm',
        'н': 'h', 'о': 'o', 'р': 'p', 'с': 'c', 'т': 't',
        'у': 'y', 'х': 'x', 'і': 'i', 'ї': 'i', 'ё': 'e'
        }  # Add more as needed
for record in df.to_dicts():
    # Apply Cyrillic to Latin conversion to all text fields
            processed_record = {}
            for col_name, value in record.items():
                if isinstance(value, str) and re.search(r'[\u0400-\u04FF]', value):
                    print(f"Row {record[no_col]} on column: {col_name} with value: {value}")
                    processed_record[col_name] = convert_cyrillic_to_latin(value)
                else:
                    processed_record[col_name] = value
            
            # Clean the 'No' field specifically for number extraction
            no_clean = processed_record[no_col]
            no_str = re.sub(r'\D', '', no_clean)
            if not no_str:
                continue  # Skip invalid records

            no = int(no_str)

            cleaned_records.append({
                'no': no,
                'provinsi': processed_record[prov_col].strip().replace('<br>', ' '),
                'kabupaten_kota': processed_record[kab_col].strip().replace('<br>', ' '),
                'nama_kota': processed_record[nama_col].strip().replace('<br>', ' '),
                'singkatan_nama_kota': processed_record[singk_col].strip().replace('<br>', ' ').upper(),
                'parent_subdivision': processed_record[parent_col].strip().replace('<br>', ' ').upper()
            })

Row  73.   on column:  Singkatan<br>Nama<br>Kota  with value:  вкт                       
Row  79.   on column:  Singkatan<br>Nama<br>Kota  with value:  ТВН                       
Row  122.  on column:  Singkatan<br>Nama<br>Kota  with value:  тві                       
Row  128.  on column:  Singkatan<br>Nama<br>Kota  with value:  ктв                       
Row  131.  on column:  Singkatan<br>Nama<br>Kota  with value:  КОТ                       
Row  149.  on column:  Singkatan<br>Nama<br>Kota  with value:  твк                       
Row  159.  on column:  Singkatan<br>Nama<br>Kota  with value:  КҮВ                       
Row  185.  on column:  Singkatan<br>Nama<br>Kota  with value:  СМН                       
Row  306.  on column:  Nama Kota              with value:  Ваа         
Row  310.  on column:  Singkatan<br>Nama<br>Kota  with value:  ТАМ                       
Row  384.  on column:  Singkatan<br>Nama<br>Kota  with value:  тмн                       
Row  423.  on column:  Singk

In [17]:
cleaned_records

[{'no': 1,
  'provinsi': 'Aceh',
  'kabupaten_kota': 'Kabupaten Aceh Selatan',
  'nama_kota': 'Tapak Tuan',
  'singkatan_nama_kota': 'TTN',
  'parent_subdivision': 'ID-AC'},
 {'no': 2,
  'provinsi': 'Aceh',
  'kabupaten_kota': 'Kabupaten Aceh Tenggara',
  'nama_kota': 'Kutacane',
  'singkatan_nama_kota': 'KTN',
  'parent_subdivision': 'ID-AC'},
 {'no': 3,
  'provinsi': 'Aceh',
  'kabupaten_kota': 'Kabupaten Aceh Timur',
  'nama_kota': 'ldi Rayeuk',
  'singkatan_nama_kota': 'IRY',
  'parent_subdivision': 'ID-AC'},
 {'no': 4,
  'provinsi': 'Aceh',
  'kabupaten_kota': 'Kabupaten Aceh Tengah',
  'nama_kota': 'Takengon',
  'singkatan_nama_kota': 'TKN',
  'parent_subdivision': 'ID-AC'},
 {'no': 5,
  'provinsi': 'Aceh',
  'kabupaten_kota': 'Kabupaten Aceh Barat',
  'nama_kota': 'Meulaboh',
  'singkatan_nama_kota': 'MBO',
  'parent_subdivision': 'ID-AC'},
 {'no': 6,
  'provinsi': 'Aceh',
  'kabupaten_kota': 'Kabupaten Aceh Besar',
  'nama_kota': 'Jantho',
  'singkatan_nama_kota': 'JTH',
  'par